# Modélisation de portefeuille et analyse de risque

Pipeline : données de marché → rendements → VaR (historique, paramétrique, Monte Carlo) → frontière efficiente → backtesting.

> Pour des données simulées hors-ligne, remplacer `download_prices()` par `simulate_prices()`.

In [ ]:
import sys; sys.path.append('..')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from src import risk
from src.data import download_prices, TICKERS

## 1. Données

In [ ]:
prices = download_prices()
returns = risk.log_returns(prices)
print(prices.index.min().date(), '->', prices.index.max().date(), '|', len(returns), 'observations')
prices.tail()

In [ ]:
(prices / prices.iloc[0] * 100).plot(figsize=(10, 4), title='Prix normalisés (base 100)');

## 2. Statistiques descriptives

In [ ]:
risk.annual_stats(returns).round(4)

In [ ]:
import matplotlib.pyplot as plt
c = returns.corr()
plt.figure(figsize=(6,5)); plt.imshow(c, cmap='RdYlGn', vmin=-1, vmax=1); plt.colorbar()
plt.xticks(range(len(c)), c.columns, rotation=45); plt.yticks(range(len(c)), c.columns); plt.title('Corrélations');

## 3. Portefeuille équipondéré : performance et risque

In [ ]:
w = np.ones(len(TICKERS)) / len(TICKERS)
r_p = risk.portfolio_returns(returns, w)
print('Rendement annuel :', round(r_p.mean()*252, 4))
print('Volatilité annuelle :', round(r_p.std()*np.sqrt(252), 4))
print('Sharpe (rf=2%) :', round(risk.sharpe_ratio(r_p, 0.02), 3))
print('Max drawdown :', round(risk.max_drawdown(r_p), 4))

## 4. Value at Risk : trois méthodes

- **Historique** : quantile empirique des rendements.
- **Paramétrique** : hypothèse de normalité (moyenne, écart-type).
- **Monte Carlo** : simulation de rendements corrélés (décomposition de Cholesky de la matrice de covariance).

In [ ]:
alpha = 0.99
sims = risk.simulate_portfolio(returns, w, horizon=1, n_sims=10_000)
res = pd.Series({
    'VaR historique': risk.var_historical(r_p, alpha),
    'VaR paramétrique': risk.var_parametric(r_p, alpha),
    'VaR Monte Carlo': -np.quantile(sims, 1-alpha),
    'ES historique': risk.es_historical(r_p, alpha),
    'VaR Monte Carlo 10j': risk.var_monte_carlo(returns, w, alpha, horizon=10),
})
res.map('{:.2%}'.format)

In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
ax.hist(r_p, bins=60, density=True, alpha=.5, label='Historique')
ax.hist(sims, bins=60, density=True, alpha=.4, label='Monte Carlo')
for k, c in [('VaR historique','red'),('VaR paramétrique','green'),('VaR Monte Carlo','purple')]:
    ax.axvline(-res[k], color=c, ls='--', label=f'{k} {res[k]:.2%}')
ax.legend(fontsize=8); ax.set_title('Distribution des rendements et VaR 99%');

**Lecture** : comparer les trois estimations. Si la VaR paramétrique est nettement plus faible que l'historique, les queues de distribution sont plus épaisses que la loi normale (kurtosis) : à commenter avec les résultats obtenus.

In [ ]:
print('Kurtosis en excès :', round(r_p.kurt(), 2), '| Skewness :', round(r_p.skew(), 2))

## 5. Frontière efficiente (long-only)

In [ ]:
frontier, w_min, w_sharpe = risk.efficient_frontier(returns)
mu = returns.mean()*252; vol = returns.std()*np.sqrt(252)
fig, ax = plt.subplots(figsize=(8,5))
ax.plot(frontier[:,0], frontier[:,1], 'b-', label='Frontière')
ax.scatter(vol, mu, c='grey')
for n in mu.index: ax.annotate(n, (vol[n], mu[n]))
f = lambda ww: (np.sqrt(ww @ (returns.cov().values*252) @ ww), ww @ mu.values)
ax.scatter(*f(w_min.values), c='green', marker='*', s=200, label='Min variance')
ax.scatter(*f(w_sharpe.values), c='red', marker='*', s=200, label='Sharpe max')
ax.set(xlabel='Volatilité', ylabel='Rendement'); ax.legend()
pd.DataFrame({'Min variance': w_min, 'Sharpe max': w_sharpe}).round(3)

## 6. Backtesting de la VaR (fenêtre glissante 250 jours)

In [ ]:
bt = risk.backtest_var(r_p, alpha)
n_exc, n_obs = int(bt['exception'].sum()), len(bt)
print(f'Exceptions : {n_exc}/{n_obs} (attendu ~{n_obs*(1-alpha):.1f})')
print('p-value Kupiec :', round(risk.kupiec_test(n_exc, n_obs, alpha), 3))

## 7. Limites

- Rendements supposés i.i.d. et paramètres estimés sur le passé.
- Monte Carlo gaussien : ne capture pas les queues épaisses.
- Pas de coûts de transaction ni de contraintes de liquidité.
- Piste d'amélioration : VaR Student-t, GARCH, Filtered Historical Simulation.